In [ ]:
import json
import pandas as pd
import numpy as np

def np_to_py(obj):
    if isinstance(obj, np.generic):
        return obj.item()
    return obj


with open("../results/domain_classification/ensembled_classification_results_original.json", "r") as f:
    data_domain_classification = json.load(f)["results"]
with open("../results/domain_classification/ensembled_editorial_classification_results_original.json") as f:
    data_editorial_classification = json.load(f)["results"]
with open("../results/classification/ensembled_classification_results_detailed_filtered.json", "r") as f:
    filtered_data_oddss = json.load(f)["results"]
with open("../results/classification/ensembled_classification_results.json", "r") as f:
    all_data_oddss = json.load(f)["results"]
with open("../results/thematic_classification/three_theme_classification_results_ensembled_three_majority_vote.json", "r") as f:
    data_thematic_classification = json.load(f)

path = "../datasets/classification_combined_df-8k-dataset-minus2025-abstracts.csv"
all_data = pd.read_csv(path)

final_data = []
for item in all_data.iloc():
    # print(item)
    abstract = item["abstract"]
    matching_odds = [d for d in all_data_oddss if d["abstract"] == abstract]
    odd = matching_odds[0]["predicted_decision"]
    if odd == "Unrelated" or odd == "Animal Study":
        primary_design = "N/A"
        secondary_designs = []
        themes = []
    else:
        index = next(i for i, d in enumerate(data_domain_classification) if d["abstract"] == abstract)
        primary_design = data_domain_classification[index]["primary_design"]
        if primary_design == "guideline_or_editorial_or_commentary":
            matching_editorials = [d for d in data_editorial_classification if d["abstract"] == abstract]
            primary_design = matching_editorials[0]["label"]
        secondary_designs = data_domain_classification[index]["secondary_designs"]
        themes = data_thematic_classification[index]["themes"]
    final_data.append({
        "Title": np_to_py(item["title"]),
        "Publication": np_to_py(item["publication"]),
        "Doi": np_to_py(item["doi"]),
        "Authors": np_to_py(item["authors"]),
        "Year": np_to_py(item["year"]),
        "Type": np_to_py(item["type"]),
        "primary_design": primary_design,
        "secondary_designs": secondary_designs,
        "themes": themes,
        "odds": odd
    }
    )
with open("../results/final_classification_results.json", "w") as f:
    json.dump( final_data, f, indent=4)

        

In [28]:
# filter final data for thme "Active Infection vs. Post-Infectious Immune Activity" andprimary design "Randomized Controlled Trial"
data = [d for d in final_data if "Active Infection vs. Post-Infectious Immune Activity" in d["themes"] and d["primary_design"] == "randomized_controlled_trial"] 
print(len(data))
with open("../results/filtered_active_infection_rct.json", "w") as f:
    json.dump( data, f, indent=4)

14
